In [2]:
"""
Scores SVM, zero-shot, and few-shot predictions against the held-out
ground truth, and produces a side-by-side comparison.

Expects each preds_*.csv to have at least: id, predicted_label
Expects eval_holdout_20_ground_truth.csv to have: id, Label

For each approach that's present, this:
  - inner-joins predictions to ground truth on id (warns if any ids don't match)
  - reports accuracy, precision/recall/F1 (macro + weighted), confusion matrix
  - saves a confusion matrix plot
Then prints/saves one summary table comparing all approaches.

Missing prediction files are skipped with a warning rather than crashing —
run this as soon as you have at least one preds_*.csv ready.
"""

import matplotlib
matplotlib.use("Agg")  # headless — no display needed
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

# ── CONFIG — edit these paths ────────────────────────────────────────────────
GROUND_TRUTH_PATH = "eval_holdout_20_ground_truth.csv"

PREDS_PATHS = {
    "SVM":       "preds_svm.csv",
    "Zero-shot": "preds_zeroshot.csv",
    "Few-shot":  "preds_fewshot.csv",
}

OUT_DIR = Path("scoring_output")
OUT_DIR.mkdir(exist_ok=True)

ID_COL = "id"
GT_LABEL_COL = "Label"
PRED_LABEL_COL = "predicted_label"


# ── Load ground truth ────────────────────────────────────────────────────────
gt_df = pd.read_csv(GROUND_TRUTH_PATH)
print(f"Ground truth: {len(gt_df)} rows from {GROUND_TRUTH_PATH}")


def score_approach(name, preds_path):
    path = Path(preds_path)
    if not path.exists():
        print(f"\n[{name}] SKIPPED — file not found: {preds_path}")
        return None

    preds_df = pd.read_csv(path)
    if PRED_LABEL_COL not in preds_df.columns:
        print(f"\n[{name}] SKIPPED — no '{PRED_LABEL_COL}' column in {preds_path}")
        return None

    merged = gt_df.merge(preds_df[[ID_COL, PRED_LABEL_COL]], on=ID_COL, how="inner")

    n_gt, n_pred, n_merged = len(gt_df), len(preds_df), len(merged)
    if n_merged < n_gt:
        missing = set(gt_df[ID_COL]) - set(preds_df[ID_COL])
        print(f"\n[{name}] WARNING: only {n_merged}/{n_gt} ground-truth ids matched. "
              f"Missing {len(missing)} ids: {list(missing)[:5]}{'...' if len(missing) > 5 else ''}")
    else:
        print(f"\n[{name}] Scored on all {n_merged} held-out rows")

    y_true = merged[GT_LABEL_COL].astype(int)
    y_pred = merged[PRED_LABEL_COL].astype(int)

    acc = accuracy_score(y_true, y_pred)
    prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    print(f"  Accuracy:          {acc:.4f}")
    print(f"  F1 (macro):        {f1_macro:.4f}")
    print(f"  F1 (weighted):     {f1_weighted:.4f}")
    print(f"  Precision (macro): {prec_macro:.4f}")
    print(f"  Recall (macro):    {rec_macro:.4f}")
    print("\n  Classification report:")
    print(classification_report(y_true, y_pred, zero_division=0))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{name} — Confusion Matrix (n={n_merged})")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    cm_path = OUT_DIR / f"confusion_matrix_{name.lower().replace(' ', '_').replace('-', '_')}.png"
    plt.savefig(cm_path, dpi=200)
    plt.close()
    print(f"  Saved: {cm_path}")

    return {
        "approach": name,
        "n_scored": n_merged,
        "n_missing": n_gt - n_merged,
        "accuracy": acc,
        "precision_macro": prec_macro,
        "recall_macro": rec_macro,
        "f1_macro": f1_macro,
        "precision_weighted": prec_weighted,
        "recall_weighted": rec_weighted,
        "f1_weighted": f1_weighted,
    }


# ── Score each approach ──────────────────────────────────────────────────────
results = []
for name, path in PREDS_PATHS.items():
    r = score_approach(name, path)
    if r is not None:
        results.append(r)

# ── Summary comparison table ─────────────────────────────────────────────────
if results:
    summary_df = pd.DataFrame(results).sort_values("f1_weighted", ascending=False)
    summary_df = summary_df.round(4)

    print("\n" + "=" * 70)
    print("SUMMARY — ranked by F1 (weighted)")
    print("=" * 70)
    print(summary_df.to_string(index=False))

    summary_path = OUT_DIR / "comparison_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"\nSaved: {summary_path}")
else:
    print("\nNo prediction files were found/scored — check PREDS_PATHS.")

Ground truth: 82 rows from eval_holdout_20_ground_truth.csv

[SVM] Scored on all 82 held-out rows
  Accuracy:          0.8415
  F1 (macro):        0.8414
  F1 (weighted):     0.8414
  Precision (macro): 0.8429
  Recall (macro):    0.8423

  Classification report:
              precision    recall  f1-score   support

           0       0.87      0.81      0.84        42
           1       0.81      0.88      0.84        40

    accuracy                           0.84        82
   macro avg       0.84      0.84      0.84        82
weighted avg       0.84      0.84      0.84        82

  Saved: scoring_output/confusion_matrix_svm.png

[Zero-shot] Scored on all 82 held-out rows
  Accuracy:          0.8293
  F1 (macro):        0.8276
  F1 (weighted):     0.8280
  Precision (macro): 0.8370
  Recall (macro):    0.8274

  Classification report:
              precision    recall  f1-score   support

           0       0.79      0.90      0.84        42
           1       0.88      0.75      0.